In [16]:
from PIL import Image
import torch
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# Define the transformation without normalization first
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to a standard size
    transforms.ToTensor(),            # Convert to tensor
])

# Load the dataset
dataset = datasets.ImageFolder('multi-types/test', transform=transform)

In [17]:
import numpy as np
import torch
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Step 1: Define initial transform (no normalization)
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Step 2: Load dataset and extract image paths
raw_dataset = datasets.ImageFolder('multi-types/data_for_test_300', transform=base_transform)
image_paths = [path for path, _ in raw_dataset.imgs]

# Step 3: Compute mean and std for normalization
def calculate_mean_std(image_paths):
    mean = np.zeros(3)
    std = np.zeros(3)
    for path in image_paths:
        image = np.array(Image.open(path).convert('RGB')) / 255.0
        for i in range(3):
            mean[i] += image[:, :, i].mean()
            std[i] += image[:, :, i].std()
    mean /= len(image_paths)
    std /= len(image_paths)
    return mean.tolist(), std.tolist()

mean, std = calculate_mean_std(image_paths)

# Step 4: Define final transform with normalization
final_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# Step 5: Reload dataset with normalized transform
dataset = datasets.ImageFolder('multi-types/data_for_test_300', transform=final_transform)
image_paths = [path for path, _ in dataset.imgs]
labels = [label for _, label in dataset.imgs]

# Step 6: Define custom dataset class
class MRIDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        image = self.transform(image) if self.transform else transforms.ToTensor()(image)
        return image, self.labels[idx]

# Step 7: Split data and create loaders
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42
)

train_dataset = MRIDataset(train_paths, train_labels, transform=final_transform)
val_dataset = MRIDataset(val_paths, val_labels, transform=final_transform)

BATCH_SIZE = 32
NUM_CLASSES = len(dataset.classes)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [18]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=7, is_ffnn=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if is_ffnn:
                images = images.view(images.size(0), -1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_losses.append(running_loss / len(train_loader.dataset))
        train_accuracies.append(100 * correct / total)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                if is_ffnn:
                    images = images.view(images.size(0), -1)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / val_total)
        val_accuracies.append(100 * val_correct / val_total)

        print(f"Epoch {epoch+1}: Train Acc {train_accuracies[-1]:.2f}%, Val Acc {val_accuracies[-1]:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies

In [19]:
import torch.nn as nn

class SimpleFFNN(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_sizes,
        output_size,
        activation=nn.ReLU,
        use_batchnorm=False,
        dropout_rate=None
    ):
        super(SimpleFFNN, self).__init__()
        layers = []
        in_features = input_size

        for idx, h in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_features, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(activation())
            if dropout_rate is not None:
                layers.append(nn.Dropout(dropout_rate))
            in_features = h

        layers.append(nn.Linear(in_features, output_size))
        self.network = nn.Sequential(*layers)

        # Logging architecture for reproducibility
        print(f"Initialized FFNN with layers: {hidden_sizes}")
        print(f"Activation: {activation.__name__}, BatchNorm: {use_batchnorm}, Dropout: {dropout_rate}")

    def forward(self, x):
        return self.network(x)

model = SimpleFFNN(input_size=150528, hidden_sizes=[128, 64], output_size=4)

Initialized FFNN with layers: [128, 64]
Activation: ReLU, BatchNorm: False, Dropout: None


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
# Define number of output classes
NUM_CLASSES = 4  # same as ResNet

# Define input size (e.g., 224x224 RGB image flattened)
input_size = 224 * 224 * 3

# Define hidden layer sizes
hidden_sizes = [128, 64]

# Instantiate your custom FFNN
ffnn_model = SimpleFFNN(
    input_size=input_size,
    hidden_sizes=hidden_sizes,
    output_size=NUM_CLASSES
).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer_ffnn = torch.optim.Adam(ffnn_model.parameters(), lr=0.001)

# Train the model
ffnn_results = train_model(
    ffnn_model,
    train_loader,
    val_loader,
    criterion,
    optimizer_ffnn,
    num_epochs=7,
    is_ffnn=True  # important: tells train_model to flatten images
)

Initialized FFNN with layers: [128, 64]
Activation: ReLU, BatchNorm: False, Dropout: None
Epoch 1: Train Acc 51.35%, Val Acc 55.42%
Epoch 2: Train Acc 72.19%, Val Acc 73.75%
Epoch 3: Train Acc 78.96%, Val Acc 75.83%
Epoch 4: Train Acc 84.58%, Val Acc 69.17%
Epoch 5: Train Acc 88.85%, Val Acc 70.83%
Epoch 6: Train Acc 93.33%, Val Acc 77.50%
Epoch 7: Train Acc 94.79%, Val Acc 74.58%


In [20]:
NUM_CLASSES = 4

ffnn_model = SimpleFFNN(input_size=150528, hidden_sizes=[128, 64], output_size=NUM_CLASSES)
optimizer_ffnn = torch.optim.Adam(ffnn_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

ffnn_results = train_model(ffnn_model, train_loader, val_loader, criterion, optimizer_ffnn, num_epochs=7, is_ffnn=True)

Initialized FFNN with layers: [128, 64]
Activation: ReLU, BatchNorm: False, Dropout: None
Epoch 1: Train Acc 54.38%, Val Acc 61.25%
Epoch 2: Train Acc 75.00%, Val Acc 61.25%
Epoch 3: Train Acc 80.31%, Val Acc 69.17%
Epoch 4: Train Acc 91.25%, Val Acc 70.42%
Epoch 5: Train Acc 93.75%, Val Acc 77.50%
Epoch 6: Train Acc 93.65%, Val Acc 74.58%
Epoch 7: Train Acc 97.71%, Val Acc 75.00%


In [25]:
NUM_CLASSES = 4  # or whatever your classification task requires

# Load pretrained ResNet18
resnet_model = models.resnet18(pretrained=True)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)
# resnet_model = resnet_model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer_resnet = torch.optim.Adam(resnet_model.parameters(), lr=0.001)

# Train the model
resnet_results = train_model(
    resnet_model,
    train_loader,
    val_loader,
    criterion,
    optimizer_resnet,
    num_epochs=7,
    is_ffnn=False
)

Epoch 1: Train Acc 76.04%, Val Acc 56.25%
Epoch 2: Train Acc 87.60%, Val Acc 83.33%
Epoch 3: Train Acc 94.58%, Val Acc 92.50%
Epoch 4: Train Acc 95.73%, Val Acc 81.25%
Epoch 5: Train Acc 97.81%, Val Acc 80.42%
Epoch 6: Train Acc 98.23%, Val Acc 89.58%
Epoch 7: Train Acc 96.04%, Val Acc 47.50%


In [26]:
import torch
from torch.optim import Optimizer

class CustomAdam(Optimizer):
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0 or not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameters: {betas}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super(CustomAdam, self).__init__(params, defaults)

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError("CustomAdam does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p.data)
                    state['exp_avg_sq'] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']

                state['step'] += 1

                if group['weight_decay'] != 0:
                    grad = grad.add(group['weight_decay'], p.data)

                # Update biased first moment estimate
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                # Update biased second raw moment estimate
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Compute bias-corrected first and second moment estimates
                denom = exp_avg_sq.sqrt().add_(group['eps'])
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                step_size = group['lr'] * (bias_correction2 ** 0.5) / bias_correction1

                # Update parameters
                p.data.addcdiv_(exp_avg, denom, value=-step_size)

        return loss

In [24]:
optimizers = {
    "Adam": torch.optim.Adam(model.parameters(), lr=0.001),
    "SGD": torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9),
    "RMSprop": torch.optim.RMSprop(model.parameters(), lr=0.001),
    "CustomAdam": CustomAdam(model.parameters(), lr=0.001)  # if you've implemented it
}

NameError: name 'CustomAdam' is not defined

In [ ]:
results_by_optimizer = {}

for name, optimizer in optimizers.items():
    print(f"\nTraining with {name} optimizer")
    
    # Reinitialize model for fair comparison
    model = SimpleFFNN(input_size=input_size, hidden_sizes=hidden_sizes, output_size=NUM_CLASSES).to(device)
    
    # Train
    train_losses, val_losses, train_accs, val_accs = train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        num_epochs=7,
        is_ffnn=True
    )
    
    # Store results
    results_by_optimizer[name] = {
        "train_accs": train_accs,
        "val_accs": val_accs,
        "train_losses": train_losses,
        "val_losses": val_losses
    }

In [ ]:
for name, results in results_by_optimizer.items():
    plt.plot(results["val_accs"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Optimizer Comparison")
plt.legend()
plt.savefig("figures/optimizer_comparison.png")